# College Holiday Job Search Agent — Self-Contained Test Runner

Run and test your full **Google ADK** multi-agent system right here in **Kaggle** (or Colab) **before** pushing to GitHub!

### Before You Click Run (Kaggle Setup — 30 seconds)
1. In the right-hand **Notebook Options** panel, switch **Internet** to **ON**.
2. In the top menu, open **Add-ons → Secrets**, add a secret named `GOOGLE_API_KEY` (paste your free [Google AI Studio API key](https://aistudio.google.com/apikey)), and check the box to attach it to this notebook.
3. Run cells **1 → 8** in order (or click **Run All**).

In [ ]:
# Cell 1: Unpack the embedded holiday-job-agent project & install dependencies (no GitHub upload needed yet!)
import base64
import io
import os
import shutil
import zipfile
from pathlib import Path

_ZIP_B64 = """
...
"""

target_parent = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
project_dir = target_parent / "holiday-job-agent"
if project_dir.exists():
    shutil.rmtree(project_dir)

raw_zip = base64.b64decode("".join(_ZIP_B64.split()))
with zipfile.ZipFile(io.BytesIO(raw_zip)) as zf:
    zf.extractall(target_parent)

os.chdir(project_dir)
print(f"Unpacked project to: {os.getcwd()}")
print("Top-level files:", sorted(p.name for p in project_dir.iterdir()))

!pip install -q -e ".[dev]"
print("Dependencies installed cleanly!")

In [ ]:
# Cell 2: Load GOOGLE_API_KEY safely & run all deterministic unit/integration tests
import getpass
import os

def _load_api_key() -> str:
    if os.environ.get("GOOGLE_API_KEY"):
        return os.environ["GOOGLE_API_KEY"]
    # 1. Try Kaggle Secrets
    try:
        from kaggle_secrets import UserSecretsClient
        val = UserSecretsClient().get_secret("GOOGLE_API_KEY")
        if val:
            print("Loaded GOOGLE_API_KEY from Kaggle Secrets.")
            return val
    except Exception:
        pass
    # 2. Try Google Colab Secrets
    try:
        from google.colab import userdata
        val = userdata.get("GOOGLE_API_KEY")
        if val:
            print("Loaded GOOGLE_API_KEY from Colab Secrets.")
            return val
    except Exception:
        pass
    # 3. Interactive fallback (input stays masked)
    return getpass.getpass("Paste your Google AI Studio GOOGLE_API_KEY: ").strip()

os.environ["GOOGLE_API_KEY"] = _load_api_key()
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "false"
os.environ["OTEL_EXPORTER_MODE"] = "none"

!pytest -v tests/unit tests/integration tests/eval/test_agent_eval.py

In [ ]:
# Cell 3: Boot the ADK Multi-Agent Runner with HITL Confirmation Support
import sys
from pathlib import Path

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from google.adk.runners import Runner
from google.genai import types

from app.agent import create_holiday_job_app
from app.config import AgentSettings, build_memory_service, build_session_service

settings = AgentSettings.from_env()
adk_app = create_holiday_job_app(settings=settings)
session_service = build_session_service(settings)
memory_service = build_memory_service(settings)

USER_ID = "student-maya"
SESSION_ID = "notebook-demo-1"

await session_service.create_session(
    app_name="app",
    user_id=USER_ID,
    session_id=SESSION_ID,
)

runner = Runner(
    app=adk_app,
    session_service=session_service,
    memory_service=memory_service,
)

pending_hitl: dict | None = None

def _render_events(event) -> None:
    global pending_hitl
    if not event.content or not event.content.parts:
        return
    for part in event.content.parts:
        if getattr(part, "function_call", None):
            fc = part.function_call
            if fc.name == "adk_request_confirmation":
                pending_hitl = {
                    "call_id": fc.id,
                    "invocation_id": event.invocation_id,
                    "args": fc.args,
                }
                print("==================================================")
                print("[HITL GATE PAUSED] Human confirmation required!")
                print("Action details:", fc.args)
                print("Run `await approve_pending_submission(True)` in the next cell to approve.")
                print("==================================================\n")
            else:
                print(f"  -> [{event.author}] calling tool: {fc.name}")
        if getattr(part, "text", None):
            print(f"[{event.author}]:\n{part.text}\n")

async def chat(message: str) -> None:
    content = types.Content(role="user", parts=[types.Part.from_text(text=message)])
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=content,
    ):
        _render_events(event)

async def approve_pending_submission(approved: bool = True) -> None:
    global pending_hitl
    if not pending_hitl:
        print("No pending Human-in-the-Loop confirmation request right now.")
        return
    req = pending_hitl
    pending_hitl = None
    reply = types.Content(
        role="user",
        parts=[
            types.Part(
                function_response=types.FunctionResponse(
                    id=req["call_id"],
                    name="adk_request_confirmation",
                    response={"confirmed": approved},
                )
            )
        ],
    )
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        invocation_id=req["invocation_id"],
        new_message=reply,
    ):
        _render_events(event)

print("ADK Multi-Agent Runner is LIVE! Run Cells 4-8 to walk through all features.")

In [ ]:
# Turn 1: Save student holiday break profile (Strict Pydantic validation + persistent state)
await chat(
    "Hi! My student ID is student-maya. I am home in ZIP 60611 from 2026-12-14 to 2027-01-12. "
    "I can commute up to 15 miles, need at least $18/hr, can work local or remote, prefer morning "
    "and weekend shifts, and my skills are customer_service, python, and tutoring. Please save my profile."
)

In [ ]:
# Turn 2: Parallel Local + Remote Scouting -> Pro Fit Ranking -> Automated Scam Blocking
await chat(
    "Search for local seasonal jobs and remote holiday gigs matching my saved profile, "
    "screen EVERY listing for scams, and shortlist the top verified matches."
)

In [ ]:
# Turn 3: Tailored Application Studio Loop (Pro Drafter + Flash Self-Eval Critic, NO submit yet)
await chat(
    "Draft a tailored application packet for LOC-101 at Northstar Outdoor Outfitters using ONLY "
    "my verified skills and break dates. Do NOT submit it yet."
)

In [ ]:
# Turn 4: Confirm submission -> Triggers ADK Human-in-the-Loop ToolConfirmation pause
await chat(
    "The draft looks great! I explicitly confirm: please submit my application for LOC-101 "
    "at Northstar Outdoor Outfitters in dry_run mode."
)

In [ ]:
# Turn 5: Approve the paused HITL confirmation gate to complete submission & get receipt!
await approve_pending_submission(True)

In [ ]:
# Optional: Run the live ADK Golden Dataset Regression Evaluation Suite!
!adk eval app tests/eval/evalsets/holiday_job_search.evalset.json --config_file_path=tests/eval/eval_config.json